# Notebook 14: Continuous Dynamics - Neural ODEs & Slow Features
**Analyzing Neural Networks as Continuous Dynamical Systems**

## What You'll Learn

In this notebook, you'll explore continuous-time perspectives on neural network dynamics:

- **Neural ODEs**: Treating networks as differential equations
- **Flow Fields**: Vector fields and phase portraits
- **Fixed Points**: Attractors and repellers in activation space
- **Slow Features**: Discovering temporal hierarchies in data
- **Characteristic Timescales**: Understanding slowness and speed

**Prerequisites**: 
- Notebooks 01 (Intro), 06 (Dynamical Systems basics)
- Basic calculus and differential equations
- Understanding of dynamical systems concepts

**Time Estimate**: 75-90 minutes

---

## Background: Why Continuous Dynamics?

### The Discrete-Continuous Connection

Standard neural networks are **discrete**: Layer 1 → Layer 2 → Layer 3

But we can also view them as **continuous differential equations**:

$$\frac{dx}{dt} = f_\theta(x(t), t)$$

Where $x(t)$ is the hidden state and $f_\theta$ is the network's velocity field.

**Why this matters**:
1. **Unified theory**: Connect to rich ODE literature
2. **Stability analysis**: Fixed points, Lyapunov functions
3. **Temporal modeling**: Natural for sequence/video data
4. **Interpretability**: Flow fields reveal computation geometry

### Key Papers

- **Neural ODEs**: Chen et al. (2018) "Neural Ordinary Differential Equations"
- **Slow Features**: Wiskott & Sejnowski (2002) "Slow Feature Analysis: Unsupervised Learning of Invariances"
- **Temporal Hierarchies**: Berkes & Wiskott (2005) "Slow feature analysis yields a rich repertoire of complex cell properties"

---

In [ ]:
# Imports
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Set random seeds
torch.manual_seed(42)
np.random.seed(42)

# Device configuration
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# Import neuros-mechint dynamics
from neuros_mechint.dynamics import (
    NeuralODEIntegrator,
    SlowFeatureAnalyzer
)

print("✓ All imports successful!")

---

## Part 1: Neural ODEs - Continuous-Time Dynamics

### From ResNets to ODEs

Consider a residual network:

$$x_{t+1} = x_t + f(x_t, \theta_t)$$

As the step size $\Delta t \rightarrow 0$, this becomes:

$$\frac{dx}{dt} = f(x(t), \theta(t))$$

This is a **neural ODE**: the hidden state evolves continuously according to a learned vector field.

### Integration Methods

To solve $\frac{dx}{dt} = f(x, t)$, we use numerical integration:

**Euler Method** (simplest):
$$x_{t+\Delta t} = x_t + \Delta t \cdot f(x_t, t)$$

**Runge-Kutta 4 (RK4)** (more accurate):
$$k_1 = f(x_t, t)$$
$$k_2 = f(x_t + \frac{\Delta t}{2}k_1, t + \frac{\Delta t}{2})$$
$$k_3 = f(x_t + \frac{\Delta t}{2}k_2, t + \frac{\Delta t}{2})$$
$$k_4 = f(x_t + \Delta t k_3, t + \Delta t)$$
$$x_{t+\Delta t} = x_t + \frac{\Delta t}{6}(k_1 + 2k_2 + 2k_3 + k_4)$$

**Dormand-Prince (dopri5)** (adaptive step size, most sophisticated)

---

In [ ]:
# Define a simple velocity field: damped oscillator
class DampedOscillator(nn.Module):
    """
    Damped harmonic oscillator:
    dx/dt = v
    dv/dt = -k*x - b*v
    
    Where k is spring constant, b is damping coefficient.
    """
    def __init__(self, k=1.0, b=0.1):
        super().__init__()
        self.k = k
        self.b = b
        
    def forward(self, state):
        # state = [x, v]
        x = state[:, 0:1]
        v = state[:, 1:2]
        
        dx_dt = v
        dv_dt = -self.k * x - self.b * v
        
        return torch.cat([dx_dt, dv_dt], dim=1)

# Create oscillator and move to device
oscillator = DampedOscillator(k=1.0, b=0.2).to(device)
print("Created damped oscillator:")
print(f"  - Spring constant k: {oscillator.k}")
print(f"  - Damping coefficient b: {oscillator.b}")
print(f"  - Device: {device}")

In [ ]:
# Initialize Neural ODE integrator
integrator = NeuralODEIntegrator(
    model=oscillator,
    dt=0.01,
    method='euler',  # Start with Euler, we'll compare methods
    verbose=True
)

print("\nNeuralODEIntegrator initialized:")
print(f"  - Time step dt: {integrator.dt}")
print(f"  - Integration method: {integrator.method}")

In [ ]:
# Integrate trajectory from initial condition
initial_state = torch.tensor([[1.0, 0.0]], device=device)  # Start at x=1, v=0
time_span = (0, 20)  # Integrate from t=0 to t=20
n_points = 500

print(f"Integrating trajectory:")
print(f"  - Initial state: x={initial_state[0, 0]:.2f}, v={initial_state[0, 1]:.2f}")
print(f"  - Time span: {time_span}")
print(f"  - Number of points: {n_points}")
print(f"  - Device: {device}\n")

result = integrator.integrate_trajectory(
    initial_state=initial_state,
    time_span=time_span,
    n_points=n_points
)

print(f"\n✓ Integration complete!")
print(f"  - Trajectory shape: {result.trajectory.shape}")
print(f"  - Times shape: {result.times.shape}")
print(f"  - Integration method: {result.method}")

In [ ]:
# Visualize trajectory
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# Extract position and velocity
times = result.times.cpu().numpy()
x = result.trajectory[:, 0].cpu().numpy()
v = result.trajectory[:, 1].cpu().numpy()

# Plot 1: Position vs time
axes[0].plot(times, x, linewidth=2, color='steelblue')
axes[0].set_xlabel('Time')
axes[0].set_ylabel('Position (x)')
axes[0].set_title('Position vs Time', fontweight='bold')
axes[0].grid(True, alpha=0.3)
axes[0].axhline(y=0, linestyle='--', color='gray', alpha=0.5)

# Plot 2: Velocity vs time
axes[1].plot(times, v, linewidth=2, color='darkorange')
axes[1].set_xlabel('Time')
axes[1].set_ylabel('Velocity (v)')
axes[1].set_title('Velocity vs Time', fontweight='bold')
axes[1].grid(True, alpha=0.3)
axes[1].axhline(y=0, linestyle='--', color='gray', alpha=0.5)

# Plot 3: Phase portrait (x vs v)
axes[2].plot(x, v, linewidth=2, color='purple', alpha=0.7)
axes[2].scatter(x[0], v[0], s=100, c='green', marker='o', 
               label='Start', zorder=5, edgecolor='black')
axes[2].scatter(x[-1], v[-1], s=100, c='red', marker='X', 
               label='End', zorder=5, edgecolor='black')
axes[2].scatter(0, 0, s=100, c='gold', marker='*', 
               label='Fixed Point', zorder=5, edgecolor='black')
axes[2].set_xlabel('Position (x)')
axes[2].set_ylabel('Velocity (v)')
axes[2].set_title('Phase Portrait', fontweight='bold')
axes[2].legend(loc='upper right')
axes[2].grid(True, alpha=0.3)
axes[2].axhline(y=0, linestyle='--', color='gray', alpha=0.3)
axes[2].axvline(x=0, linestyle='--', color='gray', alpha=0.3)

plt.tight_layout()
plt.show()

print("Observations:")
print("  - Position oscillates and decays (damping)")
print("  - Velocity leads position by π/2 (oscillator property)")
print("  - Phase portrait spirals to fixed point at (0,0)")
print("  - Fixed point is an attractor (stable equilibrium)")

### Comparing Integration Methods

Different integration methods have different:
- **Accuracy**: How close to true solution?
- **Stability**: Does it stay bounded?
- **Speed**: Computational cost

Let's compare Euler, RK4, and dopri5:

In [ ]:
# Compare integration methods
methods = ['euler', 'rk4']
results_by_method = {}

for method in methods:
    integrator_temp = NeuralODEIntegrator(
        model=oscillator,
        dt=0.05,  # Larger step to show differences
        method=method,
        verbose=False
    )
    
    result_temp = integrator_temp.integrate_trajectory(
        initial_state=initial_state,
        time_span=(0, 20),
        n_points=100
    )
    
    results_by_method[method] = result_temp
    print(f"✓ {method:10s}: integrated {len(result_temp.times)} points")

# Visualize comparison
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

colors = {'euler': 'steelblue', 'rk4': 'darkorange'}

for method, result_temp in results_by_method.items():
    times_temp = result_temp.times.cpu().numpy()
    x_temp = result_temp.trajectory[:, 0].cpu().numpy()
    v_temp = result_temp.trajectory[:, 1].cpu().numpy()
    
    # Position over time
    axes[0].plot(times_temp, x_temp, linewidth=2, 
                label=method.upper(), color=colors[method], alpha=0.8)
    
    # Phase portrait
    axes[1].plot(x_temp, v_temp, linewidth=2, 
                label=method.upper(), color=colors[method], alpha=0.8)

axes[0].set_xlabel('Time')
axes[0].set_ylabel('Position (x)')
axes[0].set_title('Integration Method Comparison - Time Series', fontweight='bold')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].set_xlabel('Position (x)')
axes[1].set_ylabel('Velocity (v)')
axes[1].set_title('Integration Method Comparison - Phase Portrait', fontweight='bold')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\nMethod Comparison:")
print("  - Euler: Simple, fast, but can drift with large dt")
print("  - RK4: More accurate, better stability, slightly slower")
print("  - For this smooth system, both methods agree well")

---

## Part 2: Flow Field Analysis

### Vector Fields and Phase Portraits

The **flow field** $f(x)$ tells us the instantaneous velocity at each point in state space:

$$\frac{dx}{dt} = f(x)$$

Visualizing this field reveals:
- **Attractors**: Where trajectories converge
- **Repellers**: Where trajectories diverge
- **Saddle points**: Stable in some directions, unstable in others
- **Limit cycles**: Periodic orbits

---

In [ ]:
# Analyze flow field
print("Analyzing flow field...\n")

flow_result = integrator.analyze_flow_field(
    state_space_bounds=(-2, 2),  # Sample from -2 to 2 in each dimension
    n_points=15  # 15x15 grid
)

print("✓ Flow field analysis complete!")
print(f"  - Flow field shape: {flow_result.flow_field.shape}")
print(f"  - Grid points shape: {flow_result.grid_points.shape}")
print(f"  - Fixed points found: {len(flow_result.fixed_points)}")

if len(flow_result.fixed_points) > 0:
    print("\nFixed Points:")
    for i, fp in enumerate(flow_result.fixed_points):
        print(f"  {i+1}. Location: {fp.cpu().numpy()}")

In [ ]:
# Visualize flow field with quiver plot
fig, ax = plt.subplots(figsize=(10, 10))

# Extract grid and flow
grid = flow_result.grid_points.cpu().numpy()
flow = flow_result.flow_field.cpu().numpy()

# Reshape for quiver plot
n_pts = int(np.sqrt(len(grid)))
X = grid[:, 0].reshape(n_pts, n_pts)
Y = grid[:, 1].reshape(n_pts, n_pts)
U = flow[:, 0].reshape(n_pts, n_pts)
V = flow[:, 1].reshape(n_pts, n_pts)

# Quiver plot (vector field)
ax.quiver(X, Y, U, V, alpha=0.6, color='steelblue')

# Overlay some trajectories - with device handling
initial_conditions = [
    torch.tensor([[1.5, 0.0]], device=device),
    torch.tensor([[0.0, 1.5]], device=device),
    torch.tensor([[-1.5, 0.0]], device=device),
    torch.tensor([[0.0, -1.5]], device=device),
]

for ic in initial_conditions:
    traj = integrator.integrate_trajectory(ic, (0, 15), n_points=200)
    x_traj = traj.trajectory[:, 0].cpu().numpy()
    v_traj = traj.trajectory[:, 1].cpu().numpy()
    ax.plot(x_traj, v_traj, linewidth=2, alpha=0.8)

# Mark fixed point
if len(flow_result.fixed_points) > 0:
    for fp in flow_result.fixed_points:
        fp_np = fp.cpu().numpy()
        ax.scatter(fp_np[0], fp_np[1], s=200, c='red', marker='*', 
                  edgecolor='black', linewidth=2, zorder=10, label='Fixed Point')

ax.set_xlabel('Position (x)', fontsize=12)
ax.set_ylabel('Velocity (v)', fontsize=12)
ax.set_title('Flow Field and Phase Portrait', fontsize=14, fontweight='bold')
ax.grid(True, alpha=0.3)
ax.set_xlim(-2, 2)
ax.set_ylim(-2, 2)
ax.legend()

plt.tight_layout()
plt.show()

print("\nInterpretation:")
print("  - Arrows point in direction of motion")
print("  - All trajectories spiral toward the fixed point")
print("  - This is a stable spiral (damped oscillator)")

---

## Part 3: More Complex Dynamics

### Nonlinear Systems and Neural Networks

Let's analyze a more complex system: a simple RNN viewed as an ODE.

**RNN Dynamics**:
$$h_{t+1} = \tanh(W_h h_t + W_x x_t)$$

As a continuous ODE:
$$\frac{dh}{dt} = \tanh(W_h h + W_x x) - h$$

---

In [ ]:
# Simple RNN as ODE
class ContinuousRNN(nn.Module):
    """RNN dynamics in continuous time."""
    def __init__(self, hidden_size=2, input_size=2):
        super().__init__()
        self.W_h = nn.Linear(hidden_size, hidden_size, bias=False)
        self.W_x = nn.Linear(input_size, hidden_size, bias=False)
        
        # Initialize with small weights for stability
        nn.init.orthogonal_(self.W_h.weight, gain=0.5)
        nn.init.xavier_uniform_(self.W_x.weight)
        
        # Constant input for autonomous dynamics
        self.register_buffer('input', torch.zeros(1, input_size))
        
    def forward(self, h):
        # Continuous RNN dynamics: dh/dt = tanh(W_h h + W_x x) - h
        new_h = torch.tanh(self.W_h(h) + self.W_x(self.input))
        return new_h - h

# Create continuous RNN and move to device
cont_rnn = ContinuousRNN(hidden_size=2, input_size=2).to(device)
print("Created Continuous RNN:")
print(f"  - Hidden size: 2")
print(f"  - Input size: 2")
print(f"  - W_h shape: {cont_rnn.W_h.weight.shape}")
print(f"  - W_x shape: {cont_rnn.W_x.weight.shape}")
print(f"  - Device: {device}")

In [ ]:
# Integrate RNN dynamics
rnn_integrator = NeuralODEIntegrator(
    model=cont_rnn,
    dt=0.01,
    method='rk4',
    verbose=False
)

# Multiple initial conditions - with proper device handling
initial_states = [
    torch.tensor([[0.8, 0.2]], device=device),
    torch.tensor([[-0.8, 0.2]], device=device),
    torch.tensor([[0.2, 0.8]], device=device),
    torch.tensor([[0.2, -0.8]], device=device),
]

# Integrate from each IC
rnn_trajectories = []
for ic in initial_states:
    traj = rnn_integrator.integrate_trajectory(ic, (0, 10), n_points=300)
    rnn_trajectories.append(traj)

print(f"✓ Integrated {len(rnn_trajectories)} trajectories")

In [ ]:
# Visualize RNN dynamics
fig = plt.figure(figsize=(12, 5))

# 2D phase portrait
ax1 = fig.add_subplot(121)
colors = plt.cm.viridis(np.linspace(0, 1, len(rnn_trajectories)))

for i, traj in enumerate(rnn_trajectories):
    h = traj.trajectory.cpu().numpy()
    ax1.plot(h[:, 0], h[:, 1], linewidth=2, color=colors[i], alpha=0.8)
    ax1.scatter(h[0, 0], h[0, 1], s=100, c=[colors[i]], marker='o', 
               edgecolor='black', zorder=5)
    ax1.scatter(h[-1, 0], h[-1, 1], s=100, c=[colors[i]], marker='X', 
               edgecolor='black', zorder=5)

ax1.set_xlabel('Hidden Dim 1', fontsize=11)
ax1.set_ylabel('Hidden Dim 2', fontsize=11)
ax1.set_title('RNN Phase Portrait', fontsize=13, fontweight='bold')
ax1.grid(True, alpha=0.3)
ax1.axhline(y=0, linestyle='--', color='gray', alpha=0.3)
ax1.axvline(x=0, linestyle='--', color='gray', alpha=0.3)

# Time series
ax2 = fig.add_subplot(122)
for i, traj in enumerate(rnn_trajectories):
    times = traj.times.cpu().numpy()
    h = traj.trajectory.cpu().numpy()
    ax2.plot(times, h[:, 0], linewidth=2, color=colors[i], alpha=0.8, label=f'IC {i+1}')

ax2.set_xlabel('Time', fontsize=11)
ax2.set_ylabel('Hidden Dim 1', fontsize=11)
ax2.set_title('Hidden State Over Time', fontsize=13, fontweight='bold')
ax2.grid(True, alpha=0.3)
ax2.legend(loc='upper right', fontsize=9)

plt.tight_layout()
plt.show()

print("\nObservations:")
print("  - RNN dynamics are more complex than linear oscillator")
print("  - Multiple attractors or complex fixed point structure possible")
print("  - Trajectories converge to stable states")

---

## Part 4: Slow Feature Analysis

### Discovering Temporal Hierarchies

Not all features vary at the same speed. **Slow Feature Analysis (SFA)** discovers features that change slowly over time, revealing temporal hierarchies.

### The SFA Objective

Find features $y = g(x)$ that:
1. **Change slowly**: Minimize $\Delta(y) = \langle \dot{y}^2 \rangle$
2. **Are decorrelated**: $\langle y_i y_j \rangle = \delta_{ij}$
3. **Have zero mean**: $\langle y \rangle = 0$
4. **Have unit variance**: $\langle y^2 \rangle = 1$

**Mathematical Formulation**:

$$\min_y \Delta(y) = \min_y \langle \dot{y}^2 \rangle$$

Subject to constraints above.

### Generalized Eigenvalue Problem

SFA reduces to solving:

$$\dot{C} w = \lambda C w$$

Where:
- $C = \langle yy^T \rangle$ (covariance matrix)
- $\dot{C} = \langle \dot{y}\dot{y}^T \rangle$ (derivative covariance)
- $w$ are the slow feature weights
- $\lambda$ are the delta values (slowness)

**Smaller $\lambda$ = slower feature**

---

In [ ]:
# Generate synthetic data with slow and fast components
def generate_mixed_timescale_data(n_timesteps=1000, dt=0.1):
    """
    Generate data with multiple timescales:
    - Slow component: low frequency sine wave
    - Fast component: high frequency sine wave
    - Noise: random fluctuations
    """
    t = np.linspace(0, n_timesteps * dt, n_timesteps)
    
    # Slow features (varying slowly)
    slow1 = np.sin(0.5 * t)  # Period ~12.5
    slow2 = np.cos(0.3 * t)  # Period ~21
    
    # Fast features (varying quickly)
    fast1 = np.sin(5 * t)    # Period ~1.25
    fast2 = np.cos(7 * t)    # Period ~0.9
    
    # Mix them to create observations
    x1 = 0.8 * slow1 + 0.2 * fast1 + 0.1 * np.random.randn(n_timesteps)
    x2 = 0.6 * slow2 + 0.3 * fast2 + 0.1 * np.random.randn(n_timesteps)
    x3 = 0.3 * slow1 + 0.5 * fast1 + 0.2 * slow2 + 0.1 * np.random.randn(n_timesteps)
    x4 = fast2 + 0.1 * slow1 + 0.1 * np.random.randn(n_timesteps)
    
    # Stack observations
    X = np.column_stack([x1, x2, x3, x4])
    
    return X, t, {'slow': np.column_stack([slow1, slow2]), 
                   'fast': np.column_stack([fast1, fast2])}

# Generate data
data, times_data, ground_truth = generate_mixed_timescale_data(n_timesteps=800)

print("Generated synthetic data:")
print(f"  - Shape: {data.shape}")
print(f"  - Time points: {len(times_data)}")
print(f"  - Contains 2 slow + 2 fast underlying features")
print(f"  - Mixed into 4 observed dimensions")

In [ ]:
# Visualize raw data
fig, axes = plt.subplots(4, 1, figsize=(14, 10), sharex=True)

for i in range(4):
    axes[i].plot(times_data, data[:, i], linewidth=1.5, alpha=0.8)
    axes[i].set_ylabel(f'X{i+1}', fontsize=11)
    axes[i].grid(True, alpha=0.3)
    axes[i].set_title(f'Observation {i+1}', fontsize=10, fontweight='bold')

axes[-1].set_xlabel('Time', fontsize=11)
plt.suptitle('Raw Observed Data (Mixed Timescales)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("\nRaw observations contain both slow and fast components mixed together.")
print("Goal: Separate slow from fast using SFA.")

In [ ]:
# Initialize Slow Feature Analyzer
sfa = SlowFeatureAnalyzer(
    expansion_degree=2,  # Use quadratic expansion
    whitening=True,      # Whiten data before SFA
    verbose=True
)

print("SlowFeatureAnalyzer initialized:")
print(f"  - Expansion degree: {sfa.expansion_degree}")
print(f"  - Whitening: {sfa.whitening}")

In [ ]:
# Run Slow Feature Analysis
print("\nRunning Slow Feature Analysis...\n")

result_sfa = sfa.analyze_timeseries(
    activations=data,
    n_slow_features=4  # Extract 4 slowest features
)

print("\n✓ SFA complete!")
print(f"\nExtracted {result_sfa.n_features} slow features")
print(f"\nDelta values (slowness metric):")
for i, delta in enumerate(result_sfa.delta_values):
    print(f"  Feature {i+1}: δ = {delta:.6f} (lower = slower)")

print(f"\nExplained slowness ratio: {result_sfa.explained_slowness_ratio:.2%}")

print(f"\nCharacteristic timescales:")
for i, tau in enumerate(result_sfa.characteristic_times):
    print(f"  Feature {i+1}: τ ≈ {tau:.2f} time units")

In [ ]:
# Visualize slow features
slow_features = result_sfa.slow_features

fig, axes = plt.subplots(4, 1, figsize=(14, 10), sharex=True)

for i in range(4):
    axes[i].plot(times_data, slow_features[:, i], linewidth=2, alpha=0.8, 
                color=f'C{i}')
    axes[i].set_ylabel(f'SF{i+1}', fontsize=11)
    axes[i].grid(True, alpha=0.3)
    axes[i].set_title(f'Slow Feature {i+1} (δ={result_sfa.delta_values[i]:.4f}, '
                     f'τ≈{result_sfa.characteristic_times[i]:.1f})', 
                     fontsize=10, fontweight='bold')

axes[-1].set_xlabel('Time', fontsize=11)
plt.suptitle('Extracted Slow Features (Ordered by Slowness)', 
            fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("\nInterpretation:")
print("  - Feature 1 has smallest δ → slowest (most slowly varying)")
print("  - Feature 4 has largest δ → fastest (still slow compared to raw data)")
print("  - SFA successfully separated temporal hierarchy!")

In [ ]:
# Compare with ground truth slow components
fig, axes = plt.subplots(2, 2, figsize=(14, 8))

# Plot ground truth slow components
axes[0, 0].plot(times_data, ground_truth['slow'][:, 0], linewidth=2, 
               label='Ground Truth Slow 1', color='black', alpha=0.7)
axes[0, 0].plot(times_data, slow_features[:, 0], linewidth=2, 
               label='Extracted SF1', color='steelblue', alpha=0.7, linestyle='--')
axes[0, 0].set_title('Slowest Feature Comparison', fontweight='bold')
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

axes[0, 1].plot(times_data, ground_truth['slow'][:, 1], linewidth=2, 
               label='Ground Truth Slow 2', color='black', alpha=0.7)
axes[0, 1].plot(times_data, slow_features[:, 1], linewidth=2, 
               label='Extracted SF2', color='darkorange', alpha=0.7, linestyle='--')
axes[0, 1].set_title('Second Slowest Feature Comparison', fontweight='bold')
axes[0, 1].legend()
axes[0, 1].grid(True, alpha=0.3)

# Plot fast components for comparison
axes[1, 0].plot(times_data[:200], ground_truth['fast'][:200, 0], linewidth=2, 
               label='Ground Truth Fast 1', color='black', alpha=0.7)
axes[1, 0].set_title('Fast Component (Not Extracted by SFA)', fontweight='bold')
axes[1, 0].legend()
axes[1, 0].grid(True, alpha=0.3)
axes[1, 0].set_xlabel('Time (zoomed)')

# Delta values
axes[1, 1].bar(range(1, 5), result_sfa.delta_values, color='steelblue', alpha=0.7)
axes[1, 1].set_xlabel('Feature Number')
axes[1, 1].set_ylabel('Delta Value (δ)')
axes[1, 1].set_title('Slowness Spectrum', fontweight='bold')
axes[1, 1].grid(True, alpha=0.3, axis='y')
axes[1, 1].set_xticks(range(1, 5))

plt.tight_layout()
plt.show()

print("\n✓ SFA successfully recovered the slow components!")
print("  Note: Exact match not expected due to rotation invariance,")
print("  but temporal characteristics (slowness) are preserved.")

### Why Slow Features Matter

**Biological Relevance**:
- Visual cortex: Complex cells respond to slowly-varying features (velocity, orientation)
- Hippocampus: Place cells encode slowly-changing location
- Temporal cortex: Object recognition relies on slow features (invariances)

**Machine Learning Applications**:
1. **Representation learning**: Discover meaningful abstractions
2. **Dimensionality reduction**: Focus on slowly-varying structure
3. **Temporal prediction**: Predict based on slow dynamics
4. **Transfer learning**: Slow features generalize better

---

## Part 5: Application to Neural Networks

### Analyzing Hidden State Dynamics

We can apply both Neural ODEs and Slow Features to understand neural network representations:

1. **Neural ODE**: Model layer-to-layer transformations as continuous flow
2. **Slow Features**: Find slowly-varying components in hidden states

This reveals:
- How information flows through the network
- Which representations are stable vs transient
- Temporal hierarchies in learned features

---

In [ ]:
# Example: Analyze RNN hidden states with SFA
print("Analyzing RNN hidden states with Slow Feature Analysis...\n")

# Collect hidden states from RNN trajectory - with device handling
long_trajectory = rnn_integrator.integrate_trajectory(
    initial_state=torch.tensor([[0.8, 0.2]], device=device),
    time_span=(0, 50),
    n_points=1000
)

rnn_hidden_states = long_trajectory.trajectory.cpu().numpy()

print(f"RNN hidden states: {rnn_hidden_states.shape}")

# Apply SFA
sfa_rnn = SlowFeatureAnalyzer(expansion_degree=1, whitening=True, verbose=False)
rnn_sfa_result = sfa_rnn.analyze_timeseries(
    activations=rnn_hidden_states,
    n_slow_features=2
)

print(f"\n✓ Extracted {rnn_sfa_result.n_features} slow features from RNN")
print(f"\nDelta values:")
for i, delta in enumerate(rnn_sfa_result.delta_values):
    print(f"  Feature {i+1}: δ = {delta:.6f}")
    
print(f"\nCharacteristic timescales:")
for i, tau in enumerate(rnn_sfa_result.characteristic_times):
    print(f"  Feature {i+1}: τ ≈ {tau:.2f} time units")

In [ ]:
# Visualize RNN slow features
fig, axes = plt.subplots(3, 1, figsize=(14, 9), sharex=True)

rnn_times = long_trajectory.times.cpu().numpy()

# Original hidden states
axes[0].plot(rnn_times, rnn_hidden_states[:, 0], linewidth=1.5, 
            label='Hidden 1', alpha=0.8)
axes[0].plot(rnn_times, rnn_hidden_states[:, 1], linewidth=1.5, 
            label='Hidden 2', alpha=0.8)
axes[0].set_ylabel('Hidden State')
axes[0].set_title('Original RNN Hidden States', fontweight='bold')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Slow feature 1
axes[1].plot(rnn_times, rnn_sfa_result.slow_features[:, 0], linewidth=2, 
            color='steelblue', alpha=0.8)
axes[1].set_ylabel('Slow Feature 1')
axes[1].set_title(f'Slowest Feature (δ={rnn_sfa_result.delta_values[0]:.4f})', 
                 fontweight='bold')
axes[1].grid(True, alpha=0.3)

# Slow feature 2
axes[2].plot(rnn_times, rnn_sfa_result.slow_features[:, 1], linewidth=2, 
            color='darkorange', alpha=0.8)
axes[2].set_ylabel('Slow Feature 2')
axes[2].set_xlabel('Time')
axes[2].set_title(f'Second Slowest Feature (δ={rnn_sfa_result.delta_values[1]:.4f})', 
                 fontweight='bold')
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\nInsights:")
print("  - Slow features capture stable modes of RNN dynamics")
print("  - These correspond to eigenmodes of the hidden state evolution")
print("  - Useful for understanding long-term behavior and stability")

---

## Summary & Key Takeaways

### What You Learned

1. **Neural ODEs**:
   - View networks as continuous differential equations
   - Integration methods: Euler, RK4, dopri5
   - Flow field analysis and phase portraits
   - Fixed point detection and stability

2. **Slow Feature Analysis**:
   - Discover slowly-varying features in temporal data
   - Generalized eigenvalue problem for slowness
   - Delta values quantify temporal hierarchy
   - Applications to representation learning

3. **Integration**:
   - Combine continuous dynamics with slow features
   - Analyze hidden state evolution in RNNs
   - Understand temporal structure of representations

### Key Equations

**Neural ODE**:
$$\frac{dx}{dt} = f_\theta(x(t), t)$$

**Euler Integration**:
$$x_{t+\Delta t} = x_t + \Delta t \cdot f(x_t, t)$$

**SFA Objective**:
$$\min_y \Delta(y) = \min_y \langle \dot{y}^2 \rangle$$

**Generalized Eigenvalue Problem**:
$$\dot{C} w = \lambda C w$$

### Practical Applications

1. **Sequence Modeling**: Understand RNN/LSTM dynamics
2. **Stability Analysis**: Detect unstable dynamics
3. **Representation Learning**: Find temporally stable features
4. **Transfer Learning**: Slow features generalize better
5. **Neuroscience**: Connect to biological systems

### Next Steps

- **Notebook 12**: Thermodynamic analysis of dynamics
- **Notebook 15**: Energy cascades and Hamiltonian decomposition
- **Notebook 16**: End-to-end pipeline integration

### References

1. **Chen et al. (2018)**: "Neural Ordinary Differential Equations"
2. **Wiskott & Sejnowski (2002)**: "Slow Feature Analysis: Unsupervised Learning of Invariances"
3. **Berkes & Wiskott (2005)**: "Slow feature analysis yields a rich repertoire of complex cell properties"
4. **Hairer et al. (2006)**: "Geometric Numerical Integration"

---

## Exercises

### Exercise 1: Van der Pol Oscillator
Implement the Van der Pol oscillator and analyze its limit cycle:
$$\frac{dx}{dt} = y$$
$$\frac{dy}{dt} = \mu(1-x^2)y - x$$

**Starter code**:

In [ ]:
# Exercise 1: Van der Pol oscillator
class VanDerPol(nn.Module):
    def __init__(self, mu=1.0):
        super().__init__()
        self.mu = mu
    
    def forward(self, state):
        # TODO: Implement Van der Pol dynamics
        pass

# TODO: Integrate and visualize limit cycle
# Your code here:
pass

### Exercise 2: Video Data SFA
Apply SFA to video frames to discover slowly-varying visual features.

**Starter code**:

In [ ]:
# Exercise 2: SFA on video
# TODO: Generate or load video frames
# TODO: Flatten frames to vectors
# TODO: Apply SFA
# TODO: Visualize slow features as images

# Your code here:
pass

### Exercise 3: Lyapunov Exponents
Compute Lyapunov exponents to measure chaos in the RNN dynamics.

**Starter code**:

In [ ]:
# Exercise 3: Lyapunov exponents
# TODO: Perturb initial condition slightly
# TODO: Integrate both trajectories
# TODO: Measure divergence over time
# TODO: Compute Lyapunov exponent

# Your code here:
pass

---

**Congratulations!** You've mastered continuous-time dynamics and temporal hierarchy discovery. You can now analyze neural networks as dynamical systems and extract slowly-varying representations.

Continue to **Notebook 15: Energy Cascades & Hamiltonian Decomposition** to understand the thermodynamic structure of these dynamics.